## Librerías

In [49]:
import os
import io
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from random import randint
from datetime import datetime
import matplotlib.pyplot as plt
from contextlib import redirect_stdout
from matplotlib.backends.backend_pdf import PdfPages

# Modelo
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.models import clone_model
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import keras
from keras import Input
from keras.regularizers import l2
from keras.optimizers import Adam
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

## 1. Configuración

In [2]:
BATCH_SIZE =    8
IMG_SIZE =      128
DROPOUT =       0.3
LEARNING_RATE = 0.0008
FactorLR =      0.7
EPOCHS =        40
PatienceES =    15
F_ACT =         'relu'
F_ACT_2 =       'sigmoid'

nombre_modelo = "CNN_kFold"
data_path = './data'

## 2. Cargar dataset

In [3]:
# Funcion para cargar imagenes y etiquetas
def cargar_imagenes_y_labels(data_path, clases):
    X = []
    y = []
    for idx, clase in enumerate(clases):
        clase_path = os.path.join(data_path, clase)
        imagenes = os.listdir(clase_path)
        for img_name in tqdm(imagenes, desc=f"Clase {clase}", unit="img"):
            img_path = os.path.join(clase_path, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                img = img.resize((IMG_SIZE, IMG_SIZE))
                img.verify()
                X.append(np.array(img))
                y.append(idx)
            except:
                os.remove(img_path)
    X = np.array(X).astype('float32') / 255.0
    # X = np.array(X).astype('float16') / 255.0
    y = np.array(y)
    return X, y

# Carga y split de datos
ruta_guardado = './npy_guardados'
os.makedirs(ruta_guardado, exist_ok=True)

ruta_X = os.path.join(ruta_guardado, f'X_{IMG_SIZE}.npy')
ruta_y = os.path.join(ruta_guardado, f'y_{IMG_SIZE}.npy')
if os.path.exists(ruta_X) and os.path.exists(ruta_y):
    print("🔄 Cargando datos desde archivos .npy...")
    X = np.load(ruta_X); print(f"✅ X cargado")
    y = np.load(ruta_y); print(f"✅ y cargado")
else:
    print("📥 Descargando data...")
    clases = ['neutral', 'sexy']
    X, y = cargar_imagenes_y_labels(data_path, clases)
    np.save(ruta_X, X); print("💾 X guardado")
    np.save(ruta_y, y); print("💾 y guardado")

🔄 Cargando datos desde archivos .npy...
✅ X cargado
✅ y cargado


## 3. Entrenamiento con Kfold

In [4]:
# Entrenamiento

rand = randint(0, 10000)
rand = 6497

# Dividir en entrenamiento(train+val)[90] y test[10]
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.1, random_state=rand, stratify=y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=rand)

# Métricas por fold
fold_results = []
all_histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval)):
    print(f"\n🔁 Fold {fold+1}/{5}")

    # Datos fold
    X_train, X_val = X_trainval[train_idx], X_trainval[val_idx]
    y_train, y_val = y_trainval[train_idx], y_trainval[val_idx]

    # Modelo
    model_fold = Sequential([
        Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input'),
        Conv2D(16, (3,3), activation=F_ACT, name='conv1'),
        MaxPooling2D(2,2, name='pool1'),

        Conv2D(32, (3,3), activation=F_ACT, name='conv2'),
        MaxPooling2D(2,2, name='pool2'),

        Conv2D(64, (3,3), activation=F_ACT, name='conv3'),
        MaxPooling2D(2,2, name='pool3'),

        Conv2D(128, (3,3), activation=F_ACT, name='conv4'),
        MaxPooling2D(2,2, name='pool4'),

        Flatten(name='flatten'),
        Dense(64, activation=F_ACT, name='dense1'),
        Dropout(DROPOUT, name='dropout'),
        Dense(1, activation=F_ACT_2, name='output')
    ])
    model_fold.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                       loss='binary_crossentropy',
                       metrics=['accuracy'])

    # Aumento de datos
    datagen = ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True
    )
    train_generator = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)

    # Callbacks
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=FactorLR,
                                  patience=2, min_lr=0.0001, verbose=1)
    early_stop = EarlyStopping(monitor='val_loss',
                                patience=PatienceES, restore_best_weights=True)

    # Entrenamiento
    history = model_fold.fit(
        train_generator,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    # Guardar historial
    all_histories.append(history.history)

    # Evaluación fold
    val_loss, val_acc = model_fold.evaluate(X_val, y_val, verbose=0)
    y_val_pred = (model_fold.predict(X_val) > 0.5).astype(int)
    report = classification_report(y_val, y_val_pred, target_names=['neutral', 'sexy'], output_dict=True)
    conf_mat = confusion_matrix(y_val, y_val_pred)

    fold_results.append({
        'fold': fold+1,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'precision': report['sexy']['precision'],
        'recall': report['sexy']['recall'],
        'f1': report['sexy']['f1-score'],
        'confusion_matrix': conf_mat,
        'history': history.history
    })

timestamp = datetime.now().strftime('%d-%m-%y_%H-%M-%S')
print(f"✅Entrenamiento terminado [{timestamp}]")


🔁 Fold 1/5
Epoch 1/40


c:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1890/1890 ━━━━━━━━━━━━━━━━━━━━ 676s 347ms/step - accuracy: 0.6248 - loss: 0.6381 - val_accuracy: 0.6997 - val_loss: 0.5635 - learning_rate: 8.0000e-04
Epoch 2/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 285s 151ms/step - accuracy: 0.7527 - loss: 0.5113 - val_accuracy: 0.8151 - val_loss: 0.3927 - learning_rate: 8.0000e-04
Epoch 3/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 321s 169ms/step - accuracy: 0.8057 - loss: 0.4199 - val_accuracy: 0.8426 - val_loss: 0.3401 - learning_rate: 8.0000e-04
Epoch 4/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 282s 148ms/step - accuracy: 0.8278 - loss: 0.3835 - val_accuracy: 0.8526 - val_loss: 0.3337 - learning_rate: 8.0000e-04
Epoch 5/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 206s 108ms/step - accuracy: 0.8449 - loss: 0.3529 - val_accuracy: 0.8661 - val_loss: 0.3077 - learning_rate: 8.0000e-04
Epoch 6/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 199s 105ms/step - accuracy: 0.8581 - loss: 0.3366 - val_accuracy: 0.8799 - val_loss: 0.2859 - learning_rate: 8.0000e-04
Epoch 7/40
1890/1890 ━━━━━━━━━━━━━━━━━━

c:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1890/1890 ━━━━━━━━━━━━━━━━━━━━ 251s 128ms/step - accuracy: 0.6305 - loss: 0.6332 - val_accuracy: 0.7608 - val_loss: 0.4949 - learning_rate: 8.0000e-04
Epoch 2/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 305s 160ms/step - accuracy: 0.7507 - loss: 0.5152 - val_accuracy: 0.8143 - val_loss: 0.4115 - learning_rate: 8.0000e-04
Epoch 3/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 294s 155ms/step - accuracy: 0.8039 - loss: 0.4393 - val_accuracy: 0.8302 - val_loss: 0.3682 - learning_rate: 8.0000e-04
Epoch 4/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 311s 163ms/step - accuracy: 0.8233 - loss: 0.3956 - val_accuracy: 0.8587 - val_loss: 0.3262 - learning_rate: 8.0000e-04
Epoch 5/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 298s 157ms/step - accuracy: 0.8381 - loss: 0.3732 - val_accuracy: 0.8579 - val_loss: 0.3308 - learning_rate: 8.0000e-04
Epoch 6/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 241s 127ms/step - accuracy: 0.8469 - loss: 0.3580 - val_accuracy: 0.8667 - val_loss: 0.3052 - learning_rate: 8.0000e-04
Epoch 7/40
1890/1890 ━━━━━━━━━━━━━━━━━━

...

1890/1890 ━━━━━━━━━━━━━━━━━━━━ 232s 119ms/step - accuracy: 0.5846 - loss: 0.6584 - val_accuracy: 0.7656 - val_loss: 0.4902 - learning_rate: 8.0000e-04
Epoch 2/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 241s 126ms/step - accuracy: 0.7667 - loss: 0.4867 - val_accuracy: 0.8241 - val_loss: 0.3944 - learning_rate: 8.0000e-04
Epoch 3/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 214s 113ms/step - accuracy: 0.8200 - loss: 0.4074 - val_accuracy: 0.8601 - val_loss: 0.3303 - learning_rate: 8.0000e-04
Epoch 4/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 209s 110ms/step - accuracy: 0.8375 - loss: 0.3693 - val_accuracy: 0.8730 - val_loss: 0.3105 - learning_rate: 8.0000e-04
Epoch 5/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 206s 108ms/step - accuracy: 0.8582 - loss: 0.3358 - val_accuracy: 0.8714 - val_loss: 0.3024 - learning_rate: 8.0000e-04
Epoch 6/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 211s 111ms/step - accuracy: 0.8641 - loss: 0.3197 - val_accuracy: 0.8802 - val_loss: 0.2913 - learning_rate: 8.0000e-04
Epoch 7/40
1890/1890 ━━━━━━━━━━━━━━━━━━

c:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1890/1890 ━━━━━━━━━━━━━━━━━━━━ 313s 157ms/step - accuracy: 0.6191 - loss: 0.6375 - val_accuracy: 0.7828 - val_loss: 0.4544 - learning_rate: 8.0000e-04
Epoch 2/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 279s 146ms/step - accuracy: 0.7662 - loss: 0.4859 - val_accuracy: 0.8283 - val_loss: 0.3794 - learning_rate: 8.0000e-04
Epoch 3/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 225s 118ms/step - accuracy: 0.7988 - loss: 0.4301 - val_accuracy: 0.8561 - val_loss: 0.3358 - learning_rate: 8.0000e-04
Epoch 4/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 210s 110ms/step - accuracy: 0.8317 - loss: 0.3836 - val_accuracy: 0.8550 - val_loss: 0.3302 - learning_rate: 8.0000e-04
Epoch 5/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 221s 116ms/step - accuracy: 0.8443 - loss: 0.3655 - val_accuracy: 0.8741 - val_loss: 0.2952 - learning_rate: 8.0000e-04
Epoch 6/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 223s 117ms/step - accuracy: 0.8560 - loss: 0.3425 - val_accuracy: 0.8548 - val_loss: 0.3425 - learning_rate: 8.0000e-04
Epoch 7/40
1890/1890 ━━━━━━━━━━━━━━━━━━

c:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1890/1890 ━━━━━━━━━━━━━━━━━━━━ 354s 185ms/step - accuracy: 0.6011 - loss: 0.6497 - val_accuracy: 0.7426 - val_loss: 0.5129 - learning_rate: 8.0000e-04
Epoch 2/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 275s 145ms/step - accuracy: 0.7561 - loss: 0.4997 - val_accuracy: 0.8024 - val_loss: 0.4349 - learning_rate: 8.0000e-04
Epoch 3/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 235s 124ms/step - accuracy: 0.8098 - loss: 0.4184 - val_accuracy: 0.8254 - val_loss: 0.3926 - learning_rate: 8.0000e-04
Epoch 4/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 215s 113ms/step - accuracy: 0.8258 - loss: 0.3904 - val_accuracy: 0.8566 - val_loss: 0.3319 - learning_rate: 8.0000e-04
Epoch 5/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 219s 116ms/step - accuracy: 0.8392 - loss: 0.3691 - val_accuracy: 0.8553 - val_loss: 0.3357 - learning_rate: 8.0000e-04
Epoch 6/40
1890/1890 ━━━━━━━━━━━━━━━━━━━━ 215s 114ms/step - accuracy: 0.8483 - loss: 0.3453 - val_accuracy: 0.8653 - val_loss: 0.3074 - learning_rate: 8.0000e-04
Epoch 7/40
1890/1890 ━━━━━━━━━━━━━━━━━━

#### Guardar modelo y resultados

In [53]:
# Crear ruta para guardar
save_path = f"./save/{nombre_modelo}/{timestamp}"
os.makedirs(save_path, exist_ok=True)
print(f"📁 Guardando resultados en {save_path}")

# Guardar modelo
model_fold.save(os.path.join(save_path, f'{nombre_modelo}_modelo.keras'))
model_fold.save(os.path.join(save_path, f"{nombre_modelo}_modelo.h5"))
model_fold.save_weights(os.path.join(save_path, f'{nombre_modelo}_pesos.weights.h5'))

📁 Guardando resultados en ./save/CNN_kFold/06-04-25_15-25-10


In [60]:
!pip install numpy==1.23.5

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
      Traceback (most recent call last):
        File "C:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
        File "C:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 137, in get_requires_for_build_wheel
          backend = _build_backend()
                    ^^^^^^^^^^^^^^^^
        File "C:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\si


  Using cached numpy-1.23.5.tar.gz (10.7 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


In [59]:
# Instala tensorflowjs si no lo tienes
# !pip install tensorflowjs

import subprocess
# Crea carpeta de salida por si no existe
output_dir = f"{save_path}/modelo_tfjs"
os.makedirs(output_dir, exist_ok=True)
modelo_h5 = os.path.join(save_path, f"{nombre_modelo}_modelo.h5")

# Convierte el modelo a formato TensorFlow.js
result = subprocess.run([
    "tensorflowjs_converter",
    "--input_format=keras",
    modelo_h5,
    output_dir
], capture_output=True, text=True)

# Mostrar la salida y errores
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)
print("Código de salida:", result.returncode)

STDOUT:

STDERR:
2025-04-06 19:10:20.266531: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-06 19:10:23.099453: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
C:\Users\cosme\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflowjs\read_weights.py:28: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  np.uint8, np.uint16, np.object, np.bool]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Use

## 4. Promedio de métricas por fold

In [61]:
df_kfold = pd.DataFrame([
    {
        "Fold": r['fold'],
        "Accuracy": round(max(r['history']['accuracy'])*100, 2),
        "Loss": round(max(r['history']['loss'])*100, 2),
        "Val_Accuracy": round(r['val_acc']*100, 2),
        "Val_Loss": round(r['val_loss']*100, 2),
        "Precision": round(r['precision']*100, 2),
        "Recall": round(r['recall']*100, 2),
        "F1-Score": round(r['f1']*100, 2)
    } for r in fold_results
])

df_kfold_mean = df_kfold.mean(numeric_only=True).round(2)
df_kfold_std = df_kfold.std(numeric_only=True).round(2)

df_kfold_val_summary = pd.DataFrame({
    "Métrica": df_kfold.columns[1:],
    "Media (%)": df_kfold_mean[1:].values,
    "Desviación (%)": df_kfold_std[1:].values
})
display(df_kfold_val_summary)

df_hiperparametros = pd.DataFrame({
    "Hiperparámetro": ["rand", "BATCH", "IMG_SIZE", "DROPOUT", "LEARNING_RATE", "FactorLR",  "EPOCHS", "PatienceES", "F_ACT", "F_ACT_2"],
    "Valor": [rand, BATCH_SIZE, IMG_SIZE, DROPOUT, LEARNING_RATE, FactorLR, EPOCHS, PatienceES, F_ACT, F_ACT_2]
})
df_hiperparametros


,Métrica,Media (%),Desviación (%)
0,Accuracy,94.78,0.37
1,Loss,59.31,0.61
2,Val_Accuracy,93.18,0.48
3,Val_Loss,18.35,1.58
4,Precision,93.19,0.41
5,Recall,93.16,0.64
6,F1-Score,93.18,0.49


,Hiperparámetro,Valor
0,rand,6497
1,BATCH,8
2,IMG_SIZE,128
3,DROPOUT,0.3
4,LEARNING_RATE,0.0008
5,FactorLR,0.7
6,EPOCHS,40
7,PatienceES,15
8,F_ACT,relu
9,F_ACT_2,sigmoid


## 5. Evaluación

In [43]:
loss, acc = model_fold.evaluate(X_test, y_test, verbose=1)
y_pred = (model_fold.predict(X_test) > 0.5).astype(int)

# Reporte y matriz de confusion
target_names = ['neutral', 'sexy']
report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
conf_mat = confusion_matrix(y_test, y_pred)

# Extraer matricas de la clase 'sexy'
precision = report['sexy']['precision']
recall = report['sexy']['recall']
f1 = report['sexy']['f1-score']

df_tests_metric = pd.DataFrame({
	"Métrica": ["Accuracy", "Loss", "Precision", "Recall", "F1-Score"],
	"Valor": [f"{round(acc*100, 		2)}%",
           	  f"{round(loss*100, 		2)}%",
           	  f"{round(precision*100, 	2)}%",
           	  f"{round(recall*100, 		2)}%",
           	  f"{round(f1*100, 			2)}%"]
})
df_tests_metric

66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9372 - loss: 0.1736
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step


,Métrica,Valor
0,Accuracy,93.1%
1,Loss,19.19%
2,Precision,93.22%
3,Recall,92.95%
4,F1-Score,93.09%


## 6. Generar informe

In [46]:
report_pdf_path = os.path.join(save_path, f'informe_fold_{nombre_modelo}.pdf')
with PdfPages(report_pdf_path) as pdf:
    verde_titulo = '#042106'  ; verde_cabecera = '#e3fde6'
    azul_titulo = '#091a5f'   ; azul_cabecera = '#e3f2fd'
    naranja_titulo = '#5f2d09'; naranja_cabecera = '#fdeee3'
    # Portada + hiperparámetros
    fig = plt.figure(figsize=(6, 3))
    ax_title = fig.add_axes([0, 0.5, 1, 0.5])
    ax_title.axis('off')
    ax_title.text(0.5, 0.7, f"Informe de Validación Cruzada - {timestamp}", fontsize=14, ha='center', va='center', color='#042106')
    
    ax_table = fig.add_axes([0.2, 0, 0.6, 0.9])
    ax_table.axis('off')
    tabla = ax_table.table(cellText=df_hiperparametros.values, colLabels=df_hiperparametros.columns,
                           cellLoc='center', loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(8)
    for i in range(len(df_hiperparametros.columns)):
        tabla[0, i].set_facecolor('#e3fde6')
    pdf.savefig(fig)
    plt.close(fig)

    # Tabla resumen promedio K-Fold
    fig = plt.figure(figsize=(6, 2.2))
    ax_title = fig.add_axes([0, 0.75, 1, 0.25])
    ax_title.axis('off')
    ax_title.text(0.5, 0.5, f"Resultados Promedio con K-Fold (k={5})", fontsize=12, ha='center', color='#091a5f')
    
    ax_table = fig.add_axes([0.15, 0, 0.7, 0.85])
    ax_table.axis('off')
    tabla = ax_table.table(cellText=df_kfold_val_summary.values, colLabels=df_kfold_val_summary.columns,
                           cellLoc='center', loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    for i in range(len(df_kfold_val_summary.columns)):
        tabla[0, i].set_facecolor('#e3f2fd')
    pdf.savefig(fig)
    plt.close(fig)

    # Gráfica de barras con métricas por fold
    fig, ax = plt.subplots(figsize=(6, 4))
    df_kfold.set_index("Fold")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind='bar', ax=ax)
    ax.set_title('Métricas por Fold')
    ax.set_ylabel('%')
    ax.set_xticklabels([f"Fold {i}" for i in df_kfold["Fold"]])
    ax.legend(loc='upper right')
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)
    
    # ================================
    # PAGINA 6: Resultados Test
    # ================================
    fig = plt.figure(figsize=(6, 2))
    ax_title = fig.add_axes([0, 0.75, 1, 0.25])
    ax_title.axis('off')
    
    ax_title.text(0.5, 0.5, f"Resultados Test [test: {len(X_test)}]", fontsize=12, ha='center', va='center', color=naranja_titulo)
    ax_table = fig.add_axes([0.25, 0, 0.5, 0.9])
    ax_table.axis('off')
    # Tabla
    tabla = ax_table.table(cellText=df_tests_metric.values, colLabels=df_tests_metric.columns, cellLoc='center', loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    for i in range(len(df_tests_metric.columns)):
        tabla[0, i].set_facecolor(naranja_cabecera)
    pdf.savefig(fig)
    plt.close(fig)

    # ================================
    # PAGINA 7: Matriz de Confusion
    # ================================
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Oranges',
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusión (test)')
    pdf.savefig(fig)
    plt.close(fig)
    
	# ================================
	# PAGINA 8: Resumen del Modelo
	# ================================
    summary_lines = []
    model_fold.summary(print_fn=lambda x: summary_lines.append(x))

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.axis('off')

    # Título
    ax.text(0.5, 1.0, "Arquitectura del Modelo", fontsize=14,
            ha='center', va='bottom', transform=ax.transAxes, color=azul_titulo)

    # Imprimir cada línea
    for i, line in enumerate(summary_lines):
        y = 0.8 - (i + 1) * (0.9 / len(summary_lines))
        ax.text(0, y, line, fontsize=8, family='monospace', transform=ax.transAxes)

    pdf.savefig(fig)
    plt.close(fig)
    
print(f"Guardado: {report_pdf_path}")

Guardado: ./save/CNN_kFold/06-04-25_15-25-10\informe_fold_CNN_kFold.pdf
